# 🍓 Strawberry Classification Project Manager

Welcome to the comprehensive project manager for your YOLO11 strawberry classification system! This notebook provides an interactive interface to manage and execute all your project scripts efficiently.

## 📋 Features
- **Dataset Management**: Prepare and analyze your strawberry datasets
- **Model Operations**: Train, validate, and run predictions
- **Visualization**: Generate insightful charts and analysis
- **Interactive Controls**: Easy-to-use interface for all operations
- **Progress Tracking**: Monitor execution status and results

## 🚀 Getting Started
Run the cells below in order to set up your environment and start managing your project!

## 📚 Import Required Libraries

In [ ]:
import os
import sys
import subprocess
import json
import yaml
import logging
from pathlib import Path
from datetime import datetime
import time

# Widget libraries for interactive interface
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output, Markdown
    WIDGETS_AVAILABLE = True
except ImportError:
    print("⚠️  ipywidgets not available. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output, Markdown
    WIDGETS_AVAILABLE = True

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ All libraries imported successfully!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version}")
print(f"📊 Widgets available: {WIDGETS_AVAILABLE}")

## ⚙️ Setup Script Directory and Path Management

In [ ]:
# Project configuration
PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
UTILS_DIR = PROJECT_ROOT / "utils"
MODELS_DIR = PROJECT_ROOT / "models"
VISUALIZATIONS_DIR = PROJECT_ROOT / "visualizations"

# Ensure directories exist
for dir_path in [SCRIPTS_DIR, CONFIGS_DIR, DATA_DIR, UTILS_DIR, MODELS_DIR, VISUALIZATIONS_DIR]:
    dir_path.mkdir(exist_ok=True)

# Add project directories to Python path
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(UTILS_DIR))

# Script information
SCRIPT_INFO = {
    "prepare_data.py": {
        "description": "🔄 Prepare datasets from raw format to train/val/test splits",
        "category": "Data Management",
        "args": ["--dataset", "--train_ratio", "--val_ratio", "--test_ratio", "--shuffle"]
    },
    "analyze_dataset.py": {
        "description": "📊 Analyze dataset and generate visualizations",
        "category": "Analysis",
        "args": ["--dataset", "--output_dir"]
    },
    "train.py": {
        "description": "🏋️ Train YOLO11 model on prepared dataset",
        "category": "Model Training",
        "args": ["--data", "--epochs", "--batch_size", "--imgsz", "--device"]
    },
    "validate.py": {
        "description": "✅ Validate trained model performance",
        "category": "Model Validation",
        "args": ["--model", "--data", "--batch_size", "--imgsz", "--device"]
    },
    "predict.py": {
        "description": "🔮 Run predictions on new images",
        "category": "Prediction",
        "args": ["--model", "--source", "--output", "--conf", "--device"]
    },
    "dataset_manager.py": {
        "description": "📋 Manage dataset configurations and metadata",
        "category": "Data Management",
        "args": ["--action", "--dataset", "--config"]
    },
    "download_model.py": {
        "description": "⬇️ Download pre-trained YOLO11 models",
        "category": "Model Management",
        "args": ["--model", "--output_dir"]
    }
}

print("🎯 Project configuration completed!")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📜 Scripts directory: {SCRIPTS_DIR}")
print(f"📊 Available scripts: {len(SCRIPT_INFO)}")

# Check which scripts actually exist
existing_scripts = []
for script_name in SCRIPT_INFO.keys():
    script_path = SCRIPTS_DIR / script_name
    if script_path.exists():
        existing_scripts.append(script_name)
        print(f"✅ {script_name}")
    else:
        print(f"❌ {script_name} - Not found")

print(f"\n🔢 Total existing scripts: {len(existing_scripts)}/{len(SCRIPT_INFO)}")

## 🔍 Create Script Discovery Function

In [ ]:
class ScriptManager:
    """Manages script discovery, execution, and monitoring."""
    
    def __init__(self, scripts_dir, script_info):
        self.scripts_dir = Path(scripts_dir)
        self.script_info = script_info
        self.execution_history = []
        
    def discover_scripts(self, category_filter=None):
        """Discover available Python scripts with optional category filtering."""
        discovered = {}
        
        for script_name, info in self.script_info.items():
            script_path = self.scripts_dir / script_name
            
            if script_path.exists():
                # Apply category filter if specified
                if category_filter and info.get("category") != category_filter:
                    continue
                    
                discovered[script_name] = {
                    "path": script_path,
                    "info": info,
                    "size": script_path.stat().st_size,
                    "modified": datetime.fromtimestamp(script_path.stat().st_mtime)
                }
        
        return discovered
    
    def get_script_categories(self):
        """Get all unique script categories."""
        categories = set()
        for info in self.script_info.values():
            categories.add(info.get("category", "Other"))
        return sorted(categories)
    
    def get_script_help(self, script_name):
        """Get help information for a specific script."""
        script_path = self.scripts_dir / script_name
        if not script_path.exists():
            return f"❌ Script {script_name} not found"
        
        try:
            # Try to get help from the script
            result = subprocess.run(
                [sys.executable, str(script_path), "--help"],
                capture_output=True,
                text=True,
                timeout=10
            )
            return result.stdout if result.stdout else result.stderr
        except subprocess.TimeoutExpired:
            return "⏱️ Script help timed out"
        except Exception as e:
            return f"❌ Error getting help: {str(e)}"

# Initialize script manager
script_manager = ScriptManager(SCRIPTS_DIR, SCRIPT_INFO)

# Discover all scripts
all_scripts = script_manager.discover_scripts()
categories = script_manager.get_script_categories()

print("🔍 Script Discovery Complete!")
print(f"📂 Categories found: {categories}")
print(f"📜 Scripts discovered: {len(all_scripts)}")

# Display scripts by category
for category in categories:
    category_scripts = script_manager.discover_scripts(category_filter=category)
    print(f"\n📁 {category} ({len(category_scripts)} scripts):")
    for script_name, details in category_scripts.items():
        print(f"  • {script_name} - {details['info']['description']}")
        print(f"    📅 Modified: {details['modified'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"    📏 Size: {details['size']:,} bytes")

## 🎮 Build Interactive Script Selector

In [ ]:
class InteractiveScriptRunner:
    """Interactive interface for running scripts with widgets."""
    
    def __init__(self, script_manager):
        self.script_manager = script_manager
        self.selected_script = None
        self.output_area = widgets.Output()
        
        # Create widgets
        self.category_dropdown = widgets.Dropdown(
            options=["All"] + script_manager.get_script_categories(),
            value="All",
            description="Category:",
            style={'description_width': 'initial'}
        )
        
        self.script_dropdown = widgets.Dropdown(
            options=[],
            description="Script:",
            style={'description_width': 'initial'}
        )
        
        self.args_text = widgets.Textarea(
            placeholder="Enter script arguments (e.g., --dataset farm_dataset --train_ratio 0.7)",
            description="Arguments:",
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%', height='100px')
        )
        
        self.run_button = widgets.Button(
            description="🚀 Run Script",
            button_style='success',
            tooltip="Execute the selected script"
        )
        
        self.help_button = widgets.Button(
            description="❓ Show Help",
            button_style='info',
            tooltip="Show script help"
        )
        
        self.clear_button = widgets.Button(
            description="🧹 Clear Output",
            button_style='warning',
            tooltip="Clear output area"
        )
        
        # Set up event handlers
        self.category_dropdown.observe(self.on_category_change, names='value')
        self.script_dropdown.observe(self.on_script_change, names='value')
        self.run_button.on_click(self.run_script)
        self.help_button.on_click(self.show_help)
        self.clear_button.on_click(self.clear_output)
        
        # Initialize script list
        self.update_script_list()
        
    def on_category_change(self, change):
        """Handle category dropdown change."""
        self.update_script_list()
        
    def on_script_change(self, change):
        """Handle script dropdown change."""
        self.selected_script = change['new']
        if self.selected_script:
            # Auto-populate common arguments
            script_info = self.script_manager.script_info.get(self.selected_script, {})
            if script_info.get('args'):
                example_args = " ".join([f"--{arg.replace('--', '')}" for arg in script_info['args'][:3]])
                self.args_text.placeholder = f"Example: {example_args}"
    
    def update_script_list(self):
        """Update the script dropdown based on selected category."""
        category = self.category_dropdown.value
        if category == "All":
            scripts = self.script_manager.discover_scripts()
        else:
            scripts = self.script_manager.discover_scripts(category_filter=category)
        
        options = [(f"{name} - {info['info']['description']}", name) 
                  for name, info in scripts.items()]
        self.script_dropdown.options = options
        
    def run_script(self, button):
        """Execute the selected script."""
        if not self.selected_script:
            with self.output_area:
                print("❌ Please select a script first!")
            return
            
        script_path = self.script_manager.scripts_dir / self.selected_script
        args = self.args_text.value.strip().split() if self.args_text.value.strip() else []
        
        with self.output_area:
            print(f"🚀 Running: {self.selected_script}")
            print(f"📝 Arguments: {' '.join(args)}")
            print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("=" * 50)
            
            try:
                # Execute script
                process = subprocess.Popen(
                    [sys.executable, str(script_path)] + args,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True,
                    bufsize=1,
                    universal_newlines=True
                )
                
                # Stream output in real-time
                for line in iter(process.stdout.readline, ''):\n                    print(line.rstrip())
                    
                process.wait()
                
                print("=" * 50)
                print(f"✅ Script completed with return code: {process.returncode}")
                print(f"⏰ Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                
                # Log execution
                self.script_manager.execution_history.append({
                    'script': self.selected_script,
                    'args': args,
                    'timestamp': datetime.now(),
                    'return_code': process.returncode
                })
                
            except Exception as e:
                print(f"❌ Error executing script: {str(e)}")
    
    def show_help(self, button):
        """Show help for the selected script."""
        if not self.selected_script:
            with self.output_area:
                print("❌ Please select a script first!")
            return
            
        with self.output_area:
            print(f"📖 Help for {self.selected_script}:")
            print("=" * 50)
            help_text = self.script_manager.get_script_help(self.selected_script)
            print(help_text)
            print("=" * 50)
    
    def clear_output(self, button):
        """Clear the output area."""
        self.output_area.clear_output()
        
    def display(self):
        """Display the interactive interface."""
        controls = widgets.VBox([
            widgets.HTML("<h3>🎮 Interactive Script Runner</h3>"),
            self.category_dropdown,
            self.script_dropdown,
            self.args_text,
            widgets.HBox([self.run_button, self.help_button, self.clear_button])
        ])
        
        return widgets.VBox([controls, self.output_area])

# Create interactive runner
runner = InteractiveScriptRunner(script_manager)
interface = runner.display()
display(interface)

## ⚡ Quick Action Buttons

In [ ]:
class QuickActions:
    """Quick action buttons for common workflows."""
    
    def __init__(self, script_manager):
        self.script_manager = script_manager
        self.quick_output = widgets.Output()
        
    def create_quick_buttons(self):
        """Create quick action buttons for common tasks."""
        
        # Common workflow buttons
        prepare_data_btn = widgets.Button(
            description="📊 Prepare Dataset",
            button_style='primary',
            tooltip="Prepare farm_dataset with default 70/20/10 split"
        )
        
        analyze_data_btn = widgets.Button(
            description="📈 Analyze Dataset",
            button_style='info',
            tooltip="Generate dataset analysis and visualizations"
        )
        
        train_model_btn = widgets.Button(
            description="🏋️ Train Model",
            button_style='success',
            tooltip="Train YOLO11 model with default settings"
        )
        
        validate_model_btn = widgets.Button(
            description="✅ Validate Model",
            button_style='warning',
            tooltip="Validate trained model performance"
        )
        
        # Utility buttons
        check_setup_btn = widgets.Button(
            description="🔧 Check Setup",
            button_style='secondary',
            tooltip="Check project setup and dependencies"
        )
        
        view_results_btn = widgets.Button(
            description="📋 View Results",
            button_style='secondary',
            tooltip="View recent execution results"
        )
        
        # Bind click events
        prepare_data_btn.on_click(self.prepare_dataset)
        analyze_data_btn.on_click(self.analyze_dataset)
        train_model_btn.on_click(self.train_model)
        validate_model_btn.on_click(self.validate_model)
        check_setup_btn.on_click(self.check_setup)
        view_results_btn.on_click(self.view_results)
        
        # Layout buttons
        workflow_buttons = widgets.HBox([
            prepare_data_btn,
            analyze_data_btn,
            train_model_btn,
            validate_model_btn
        ])
        
        utility_buttons = widgets.HBox([
            check_setup_btn,
            view_results_btn
        ])
        
        return widgets.VBox([
            widgets.HTML("<h3>⚡ Quick Actions</h3>"),
            widgets.HTML("<p><b>🔄 Workflow:</b> Common data science workflow buttons</p>"),
            workflow_buttons,
            widgets.HTML("<p><b>🛠️ Utilities:</b> Project management and monitoring</p>"),
            utility_buttons,
            self.quick_output
        ])
    
    def execute_script_async(self, script_name, args=[]):
        """Execute a script with given arguments."""
        script_path = self.script_manager.scripts_dir / script_name
        
        if not script_path.exists():
            with self.quick_output:
                print(f"❌ Script {script_name} not found!")
            return
        
        with self.quick_output:
            print(f"🚀 Executing: {script_name}")
            print(f"📝 Arguments: {' '.join(args)}")
            print(f"⏰ Started: {datetime.now().strftime('%H:%M:%S')}")
            
            try:
                result = subprocess.run(
                    [sys.executable, str(script_path)] + args,
                    capture_output=True,
                    text=True,
                    timeout=300  # 5 minute timeout
                )
                
                if result.stdout:
                    print(f"📤 Output:\\n{result.stdout}")
                if result.stderr:
                    print(f"⚠️  Errors:\\n{result.stderr}")
                    
                if result.returncode == 0:
                    print(f"✅ {script_name} completed successfully!")
                else:
                    print(f"❌ {script_name} failed with return code {result.returncode}")
                    
            except subprocess.TimeoutExpired:
                print(f"⏱️ {script_name} timed out after 5 minutes")
            except Exception as e:
                print(f"❌ Error executing {script_name}: {str(e)}")
            
            print("=" * 50)
    
    def prepare_dataset(self, button):
        """Quick action: Prepare dataset with default settings."""
        self.execute_script_async("prepare_data.py", [
            "--dataset", "farm_dataset",
            "--train_ratio", "0.7",
            "--val_ratio", "0.2",
            "--test_ratio", "0.1",
            "--shuffle"
        ])
    
    def analyze_dataset(self, button):
        """Quick action: Analyze dataset."""
        self.execute_script_async("analyze_dataset.py", [
            "--dataset", "farm_dataset"
        ])
    
    def train_model(self, button):
        """Quick action: Train model with default settings."""
        self.execute_script_async("train.py", [
            "--data", "configs/farm_dataset_data.yaml",
            "--epochs", "50",
            "--batch_size", "16",
            "--imgsz", "640"
        ])
    
    def validate_model(self, button):
        """Quick action: Validate model."""
        self.execute_script_async("validate.py", [
            "--model", "models/best.pt",
            "--data", "configs/farm_dataset_data.yaml"
        ])
    
    def check_setup(self, button):
        """Quick action: Check project setup."""
        with self.quick_output:
            print("🔧 Checking Project Setup...")
            print("=" * 50)
            
            # Check directories
            dirs_to_check = [
                ("Scripts", SCRIPTS_DIR),
                ("Configs", CONFIGS_DIR),
                ("Data", DATA_DIR),
                ("Models", MODELS_DIR),
                ("Utils", UTILS_DIR)
            ]
            
            for name, path in dirs_to_check:
                status = "✅" if path.exists() else "❌"
                print(f"{status} {name}: {path}")
            
            # Check scripts
            print("\\n📜 Script Status:")
            for script_name in SCRIPT_INFO.keys():
                script_path = SCRIPTS_DIR / script_name
                status = "✅" if script_path.exists() else "❌"
                print(f"{status} {script_name}")
            
            # Check datasets
            print("\\n📊 Dataset Status:")
            datasets_dir = DATA_DIR / "datasets"
            if datasets_dir.exists():
                for dataset_dir in datasets_dir.iterdir():
                    if dataset_dir.is_dir():
                        print(f"📁 {dataset_dir.name}")
                        for subdir in ["images", "labels"]:
                            subpath = dataset_dir / subdir
                            if subpath.exists():
                                count = len(list(subpath.glob("*")))
                                print(f"  📂 {subdir}: {count} files")
            
            print("=" * 50)
    
    def view_results(self, button):
        """Quick action: View execution history."""
        with self.quick_output:
            print("📋 Execution History:")
            print("=" * 50)
            
            if not self.script_manager.execution_history:
                print("📝 No execution history available")
                return
            
            for i, entry in enumerate(self.script_manager.execution_history[-10:], 1):
                status = "✅" if entry['return_code'] == 0 else "❌"
                print(f"{i:2d}. {status} {entry['script']}")
                print(f"    📅 {entry['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"    📝 Args: {' '.join(entry['args'])}")
                print(f"    🔢 Return code: {entry['return_code']}")
                print()
            
            print("=" * 50)

# Create and display quick actions
quick_actions = QuickActions(script_manager)
quick_interface = quick_actions.create_quick_buttons()
display(quick_interface)

## 📊 Project Monitoring & Status

In [ ]:
class ProjectMonitor:
    """Monitor project status and provide insights."""
    
    def __init__(self, project_root):
        self.project_root = Path(project_root)
        
    def get_project_stats(self):
        """Get comprehensive project statistics."""
        stats = {}
        
        # Dataset statistics
        datasets_dir = self.project_root / "data" / "datasets"
        if datasets_dir.exists():
            stats['datasets'] = {}
            for dataset_dir in datasets_dir.iterdir():
                if dataset_dir.is_dir():
                    dataset_stats = {}
                    
                    # Count images
                    images_dir = dataset_dir / "images"
                    if images_dir.exists():
                        image_files = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.jpeg")) + list(images_dir.glob("*.png"))
                        dataset_stats['images'] = len(image_files)
                    
                    # Count labels
                    labels_dir = dataset_dir / "labels"
                    if labels_dir.exists():
                        label_files = list(labels_dir.glob("*.txt"))
                        dataset_stats['labels'] = len(label_files)
                    
                    # Get classes
                    classes_file = dataset_dir / "classes.txt"
                    if classes_file.exists():
                        with open(classes_file, 'r') as f:
                            classes = [line.strip() for line in f.readlines()]
                            dataset_stats['classes'] = len(classes)
                            dataset_stats['class_names'] = classes
                    
                    stats['datasets'][dataset_dir.name] = dataset_stats
        
        # Model statistics
        models_dir = self.project_root / "models"
        if models_dir.exists():
            model_files = list(models_dir.glob("*.pt"))
            stats['models'] = len(model_files)
            stats['model_files'] = [m.name for m in model_files]
        
        # Visualization statistics
        viz_dir = self.project_root / "visualizations"
        if viz_dir.exists():
            viz_files = list(viz_dir.glob("*.png")) + list(viz_dir.glob("*.jpg"))
            stats['visualizations'] = len(viz_files)
        
        return stats
    
    def display_dashboard(self):
        """Display project monitoring dashboard."""
        stats = self.get_project_stats()
        
        # Create dashboard HTML
        dashboard_html = f\"\"\"
        <div style="background-color: #f8f9fa; padding: 20px; border-radius: 10px; margin: 10px 0;">
            <h3 style="color: #2c3e50; margin-bottom: 20px;">📊 Project Dashboard</h3>
            
            <div style="display: flex; flex-wrap: wrap; gap: 20px;">
                <!-- Dataset Stats -->
                <div style="flex: 1; min-width: 300px; background: white; padding: 15px; border-radius: 8px; border-left: 4px solid #3498db;">
                    <h4 style="color: #3498db; margin-bottom: 10px;">📁 Datasets</h4>
                    {self._generate_dataset_stats_html(stats.get('datasets', {}))}
                </div>
                
                <!-- Model Stats -->
                <div style="flex: 1; min-width: 300px; background: white; padding: 15px; border-radius: 8px; border-left: 4px solid #e74c3c;">
                    <h4 style="color: #e74c3c; margin-bottom: 10px;">🤖 Models</h4>
                    <p><strong>Total Models:</strong> {stats.get('models', 0)}</p>
                    {self._generate_model_list_html(stats.get('model_files', []))}
                </div>
                
                <!-- Visualization Stats -->
                <div style="flex: 1; min-width: 300px; background: white; padding: 15px; border-radius: 8px; border-left: 4px solid #f39c12;">
                    <h4 style="color: #f39c12; margin-bottom: 10px;">📈 Visualizations</h4>
                    <p><strong>Generated Plots:</strong> {stats.get('visualizations', 0)}</p>
                </div>
            </div>
            
            <div style="margin-top: 20px; padding: 15px; background: white; border-radius: 8px;">
                <h4 style="color: #27ae60;">🎯 Quick Tips</h4>
                <ul style="margin-left: 20px;">
                    <li>Use <strong>Prepare Dataset</strong> to create train/val/test splits</li>
                    <li>Run <strong>Analyze Dataset</strong> to understand your data distribution</li>
                    <li>Start with <strong>Train Model</strong> using default settings</li>
                    <li>Always <strong>Validate Model</strong> after training</li>
                </ul>
            </div>
        </div>
        \"\"\"
        
        display(HTML(dashboard_html))
    
    def _generate_dataset_stats_html(self, datasets):
        \"\"\"Generate HTML for dataset statistics.\"\"\"
        if not datasets:
            return "<p><em>No datasets found</em></p>"
        
        html = ""
        for name, stats in datasets.items():
            html += f\"\"\"
            <div style="margin-bottom: 15px; padding: 10px; background: #f8f9fa; border-radius: 5px;">
                <h5 style="color: #2c3e50; margin-bottom: 5px;">{name}</h5>
                <p style="margin: 2px 0;"><strong>Images:</strong> {stats.get('images', 0):,}</p>
                <p style="margin: 2px 0;"><strong>Labels:</strong> {stats.get('labels', 0):,}</p>
                <p style="margin: 2px 0;"><strong>Classes:</strong> {stats.get('classes', 0)}</p>
                {self._generate_classes_html(stats.get('class_names', []))}
            </div>
            \"\"\"
        return html
    
    def _generate_classes_html(self, classes):
        \"\"\"Generate HTML for class list.\"\"\"
        if not classes:
            return ""
        
        classes_str = ", ".join(classes[:5])  # Show first 5 classes
        if len(classes) > 5:
            classes_str += f" ... (+{len(classes) - 5} more)"
        
        return f'<p style="margin: 2px 0; font-size: 0.9em; color: #666;"><em>Classes: {classes_str}</em></p>'
    
    def _generate_model_list_html(self, model_files):
        \"\"\"Generate HTML for model file list.\"\"\"
        if not model_files:
            return "<p><em>No model files found</em></p>"
        
        html = "<ul style='margin-left: 20px;'>"
        for model in model_files[:5]:  # Show first 5 models
            html += f"<li>{model}</li>"
        if len(model_files) > 5:
            html += f"<li><em>... and {len(model_files) - 5} more</em></li>"
        html += "</ul>"
        return html

# Create and display project monitor
monitor = ProjectMonitor(PROJECT_ROOT)
monitor.display_dashboard()

## 🎉 Ready to Go!

Your comprehensive project manager is now set up! Here's how to use it:

### 🔄 Typical Workflow:
1. **📊 Prepare Dataset**: Use the quick action button or interactive runner to prepare your farm_dataset
2. **📈 Analyze Dataset**: Generate visualizations and understand your data distribution
3. **🏋️ Train Model**: Start training with default settings or customize parameters
4. **✅ Validate Model**: Check model performance on validation set
5. **🔮 Make Predictions**: Run inference on new images

### 🎮 Interactive Features:
- **Category Filter**: Filter scripts by category (Data Management, Model Training, etc.)
- **Real-time Output**: See script execution in real-time
- **Help System**: Get help for any script with the help button
- **Execution History**: Track all script runs and their results

### ⚡ Quick Actions:
- **One-click workflows**: Common tasks with sensible defaults
- **Project monitoring**: Real-time project status and statistics
- **Setup verification**: Check that all components are properly configured

### 💡 Pro Tips:
- Run cells in order the first time to set up all components
- Use the **Clear Output** button to keep the interface clean
- Check the **Project Dashboard** regularly to monitor progress
- The **Execution History** helps track what you've done

**Happy coding! 🚀**